# 25. 行列操作与类型转换

<!-- module-learning-arc:start -->
> **Pandas 模块主线｜第 4 / 10 步：修正类型并建立干净字段**
>
> **持续应用背景：** 搭建电商履约异常追踪台：把订单、客户、商品和履约信息整理成安全合并的事实表，再生成趋势指标和异常工单。
>
> **承接上一阶段：** 选择、筛选与排序  →  **本章任务：** 行列操作与类型转换  →  **下一步：** 数据质量检查与清洗
>
> **大作业连接：** 本章练习将成为《电商履约异常追踪台》的一部分，最终需要从多表质量审计走到订单粒度事实表、窗口趋势和可复核异常工单。
<!-- module-learning-arc:end -->


## 本章场景

清洗数据时最常做的动作，就是改列、加列、删列，以及把一列从文本变成数值。



## 本章目标

学完本章，你将能够：

- **理解**：理解列的新增/删除/rename、stack/unstack 与 astype。
- **操作**：能新增列、改列名、转换数据类型。
- **迁移**：能把一张经营表整理成字段规范、类型正确的分析表。


## 25.1 核心概念

**背景引入**：清洗数据时最常做的动作，就是改列、加列、删列，以及把一列从文本变成数值。真实报表里的金额经常是字符串，订单状态里夹着“取消”要单独处理，不会这两招，计算平均值和汇总时就会卡壳。把它们理顺了，一张“原始表”就能变成一张“能直接算的表”。

- 派生列应记录计算口径。
- 链式赋值可能只修改临时对象，优先使用loc。
- 类型转换失败时应选择报错或转为缺失值。

> **直观类比**：对一张表加列，就像在台账上加一栏“备注”，得记在正本上；链式赋值却可能让你改的是一份“复印件”——正本纹丝不动，所以跳出的 SettingWithCopyWarning 就是“你好像在改复印件”的提醒，改用 loc 就能把字落回正本。


## 25.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| assign() | `pd.DataFrame()`、`orders.assign()`、`orders['amount']`、`orders['quantity']` | assign返回包含新列的新表，适合连续处理。 | 触发SettingWithCopyWarning仍继续运行 |
| loc 条件更新 | `pd.DataFrame()`、`loc[orders['status']` | loc可以只更新满足条件的行，避免链式赋值。 | 直接覆盖原始列却没有保留转换前数据 |
| drop() | `pd.DataFrame()`、`data.drop()` | drop默认返回新对象，不会自动改变原表。 | errors='coerce'后不检查新增缺失值 |
| rename() | `pd.DataFrame()`、`data.rename()` | 重命名后列名更能表达业务含义。 | 触发SettingWithCopyWarning仍继续运行 |
| to_numeric() | `pd.Series()`、`pd.to_numeric()`、`parsed.isna()`、`.sum()` | errors='coerce'会把无法解析的值转换为缺失值，需要后续检查。 | 直接覆盖原始列却没有保留转换前数据 |
| astype() | `pd.DataFrame()`、`.astype()`、`data['region']` | 分类字段可以转换为category，明确字段语义。 | errors='coerce'后不检查新增缺失值 |


## 25.3 示例 1：新增与修改列

**背景引入**：订单表里只有金额和数量，可要分析“单价”就得自己算出一列；更麻烦的是，碰到“取消”的订单想把金额清零。给表加新列、按条件改旧列，是清洗时最先想动手的两件事。

**讲解**：`assign` 用来链式生成带新列的新表，`loc` 用来按条件精准更新某些行的值。

- `orders.assign(unit_price=orders["amount"]/orders["quantity"])`：基于现有列算新列，整列一起算；
- `orders.loc[orders["status"]=="取消", "amount"] = 0`：只把“取消”那一行的金额改成 0，其他行不动；
- 条件更新务必用 `loc` 而不是链式赋值，避免只改了临时对象还弹出警告；
- **口诀**：加列用 assign，按条件改值用 loc，改了别忘了留一份改之前的数据。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "amount": [320, 880, 460, 1250],
        "quantity": [2, 4, 1, 5],
        "status": ["完成", "完成", "取消", "完成"],
    }
)
orders = orders.assign(unit_price=orders["amount"] / orders["quantity"])
orders.loc[orders["status"] == "取消", "amount"] = 0
print(orders)


## 25.4 示例 2：删除与重命名

**背景引入**：订单表里躺着不需要的“status”列，列名 `amount`、`quantity` 又太含糊，拿去给同事看要解释半天。删掉无用的列、把列名改得能看懂，表才能交付给别人用。

**讲解**：`drop` 删行列、`rename` 改列名，两者默认都**返回新对象**，不会偷偷改原表，这正是它们安全的地方。

- `orders.drop(columns=["status"])`：删掉指定列，要留着就在这行继续往下连写；
- `.rename(columns={"amount":"sales_amount", "quantity":"item_count"})`：把生硬列名改成能表达业务含义的名字；
- `.drop(index=2).reset_index(drop=True)`：删掉一行后把索引用整数重排，避免留下空洞的标签；
- **口诀**：drop 删列、rename 改名，都返回新表不碰原件，想原地改才用 inplace。


In [ ]:
clean = orders.drop(columns=["status"]).rename(
    columns={
        "amount": "sales_amount",
        "quantity": "item_count",
    }
)
clean = clean.drop(index=2).reset_index(drop=True)
print(clean)


## 25.5 示例 3：类型转换

**背景引入**：从 Excel 导入的金额经常是 `"320.5"` 这样的字符串，甚至还混着 `"N/A"`。字符串不能直接求和、求平均；地区列存成字符串又占内存。先确认类型、再安全地转成数值或分类，是算账前的必经一站。

**讲解**：`to_numeric` 把文本转数值，`astype` 把明确字段转成分类类型；转数值时用 `errors` 决定遇到坏值的态度。

- `pd.to_numeric(raw["amount"], errors="coerce")`：`"320.5"`、`"880"` 变成数字，`"N/A"` 变成 `NaN` 而不是报错；
- 转完立刻 `raw["amount"].isna().sum()` 数一下新增了多少缺失，别在缺失没查清前就往下算；
- `raw["region"].astype("category")`：把地区转成分类类型，明确“这几个取值”，还省内存；
- **口诀**：to_numeric 把文本变数字，errors=coerce 坏值变 NaN，转完先数缺失再动工。


In [ ]:
raw = pd.DataFrame(
    {
        "amount": ["320.5", "N/A", "880"],
        "region": ["华东", "华南", "华东"],
    }
)
raw["amount"] = pd.to_numeric(raw["amount"], errors="coerce")
raw["region"] = raw["region"].astype("category")
print(raw)
print(raw.dtypes)


## 25.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# assign()
# assign返回包含新列的新表，适合连续处理。
import pandas as pd

orders = pd.DataFrame({"amount": [320, 880], "quantity": [2, 4]})
result = orders.assign(unit_price=orders["amount"] / orders["quantity"])
print(result)


In [ ]:
# loc 条件更新
# loc可以只更新满足条件的行，避免链式赋值。
import pandas as pd

orders = pd.DataFrame(
    {"amount": [320, 880, 460], "status": ["完成", "取消", "完成"]}
)
orders.loc[orders["status"] == "取消", "amount"] = 0
print(orders)


In [ ]:
# drop()
# drop默认返回新对象，不会自动改变原表。
import pandas as pd

data = pd.DataFrame(
    {"id": ["A1", "A2"], "amount": [320, 880], "temporary": [1, 2]}
)
clean = data.drop(columns=["temporary"])
print(clean)


In [ ]:
# rename()
# 重命名后列名更能表达业务含义。
import pandas as pd

data = pd.DataFrame({"amount": [320, 880], "qty": [2, 4]})
renamed = data.rename(columns={"amount": "sales_amount", "qty": "item_count"})
print(renamed)


In [ ]:
# to_numeric()
# errors='coerce'会把无法解析的值转换为缺失值，需要后续检查。
import pandas as pd

raw = pd.Series(["320.5", "N/A", "880"])
parsed = pd.to_numeric(raw, errors="coerce")
print(parsed)
print("缺失数:", parsed.isna().sum())


In [ ]:
# astype()
# 分类字段可以转换为category，明确字段语义。
import pandas as pd

data = pd.DataFrame({"region": ["华东", "华南", "华东"]})
data["region"] = data["region"].astype("category")
print(data.dtypes)


**练一练 21.6**：有一张订单表，含 amount（金额，字符串）、quantity（数量）、status（状态）三列。完成三件事：① 用 assign 新增一列 unit_price = amount ÷ quantity（先把 amount 用 pd.to_numeric 转成数值再除，结果保留两位小数即可用数值体现）；② 用 loc 把 status 为“取消”的那一行 amount 改成 0；③ 用 rename 把列 amount 改名为 sales，再把它转成数值类型，并打印 sales 列的 dtype 和整张表。数据用下面这 3 行订单即可。


In [ ]:
# 请在下方填写代码
import pandas as pd

orders = pd.DataFrame(
    {
        "amount": ["320.5", "880", "460"],
        "quantity": [2, 4, 2],
        "status": ["完成", "取消", "完成"],
    }
)

# TODO ①: 用 assign 新增一列 unit_price = amount ÷ quantity
# 提示：先把 amount 用 pd.to_numeric 转成数值再相除，结果保存在 orders2，并保存第一行的单价为
# unit_price_first

# TODO ②: 用 loc 把 status 为“取消”的那一行 amount 改成 0（在 orders2 上操作，结果仍保存在 orders2）

# TODO ③: 用 rename 把列 amount 改名为 sales，再用 astype / to_numeric 转为数值，打印
# sales 的 dtype 和整张表


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "amount": ["320.5", "880", "460"],
        "quantity": [2, 4, 2],
        "status": ["完成", "取消", "完成"],
    }
)

# ① 新增列 unit_price = amount ÷ quantity，并把第一行单价存进 unit_price_first
orders2 = orders.assign(
    unit_price=pd.to_numeric(orders["amount"]) / orders["quantity"]
)
unit_price_first = float(orders2["unit_price"].iloc[0])

# ② 用 loc 把“取消”行的 amount 改成 0
orders2.loc[orders2["status"] == "取消", "amount"] = 0

# ③ 重命名并转数值
orders2 = orders2.rename(columns={"amount": "sales"})
orders2["sales"] = pd.to_numeric(orders2["sales"])
print(orders2.dtypes)
print(orders2)


## 25.7 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Description": "string",
        "Country": "category",
    },
).rename(
    columns={
        "InvoiceNo": "order_id",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "order_time",
        "UnitPrice": "unit_price",
        "CustomerID": "customer_id",
        "Country": "country",
    }
)
large_orders["sales"] = (
    large_orders["quantity"] * large_orders["unit_price"]
).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C")
    | (large_orders["quantity"] < 0),
    "取消/退货",
    "完成",
)
print("UCI Online Retail 公开数据：")
print(f"  {len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print(
    "内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
)
large_orders.head()


In [ ]:
optimized = large_orders.copy()
before_mb = optimized.memory_usage(deep=True).sum() / 1024**2
for column in ["country", "status"]:
    optimized[column] = optimized[column].astype("category")
after_mb = optimized.memory_usage(deep=True).sum() / 1024**2
print(f"类型优化前：{before_mb:.1f} MB")
print(f"类型优化后：{after_mb:.1f} MB")
print(f"节省：{(1 - after_mb / before_mb):.1%}")
print(optimized.dtypes)


## 25.8 独立迁移练习

替换一个字段或分组口径，并核对处理前后的行数与粒度。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 25.9 本章实训：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南"],
        "channel": ["线上", "线下", "线上", "线下"],
        "sales": [120, 80, 150, 100],
    }
)
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 25.9.1 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。



In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 25.9.2 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。



## 25.10 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 25.10.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。



## 25.11 易错点提醒

- 触发SettingWithCopyWarning仍继续运行
- 直接覆盖原始列却没有保留转换前数据
- errors='coerce'后不检查新增缺失值


## 25.12 练习与作业

1. 创建销售额和成本列
2. 计算利润和利润率
3. 把地区转换为分类类型

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 25.13 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“创建销售额和成本列”。
2. **独立完成**：不复制示例代码，完成“计算利润和利润率”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“把地区转换为分类类型”，用一两句话说明你修改了什么。

### 25.13.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 25.13.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

# TODO: 计算利润
# TODO: 计算利润率
# TODO: 把地区转换为分类类型
# TODO：请在下方完成 —— 21.13 练习与作业 1. 创建销售额和成本列 2. 计算利润和利润率 3. 把地区转换为分类类型 提交前检查：代码可


In [ ]:
import pandas as pd

finance = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北"],
        "sales": [1280, 960, 1100],
        "cost": [820, 710, 760],
    }
)
finance["profit"] = finance["sales"] - finance["cost"]
finance["margin"] = finance["profit"] / finance["sales"]
finance["region"] = finance["region"].astype("category")
print(finance)
print(finance.dtypes)


## 25.14 小结

系统掌握新增、修改、删除、重命名和类型转换。

**迁移思考**：

1. 如果需要批量修改多个条件下的值（如取消订单的金额改为0，退款订单的金额改为负数），应该如何组织代码？
2. 为什么使用 errors='coerce' 后要检查新增的缺失值？这些缺失值代表什么？



### 25.14.1 你已经掌握

- 新增派生列
- 安全更新数据
- 删除和重命名行列
- 转换数值与分类类型



### 25.14.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。



### 25.14.3 需要注意

- 触发SettingWithCopyWarning仍继续运行
- 直接覆盖原始列却没有保留转换前数据
- errors='coerce'后不检查新增缺失值



### 25.14.4 完成检查

- [ ] 能够新增派生列
- [ ] 能够安全更新数据
- [ ] 能够删除和重命名行列
- [ ] 能够转换数值与分类类型



### 25.14.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。

